In [1]:
pip install -U langchainhub

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\soobi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
pip install -U langchainhub langchain-openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\soobi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from typing_extensions import List, TypedDict
from langchain_core.documents import Document
from langgraph.graph import StateGraph

class AgentState(TypedDict):
    query: str
    context: List[Document]
    answer: str
    
graph_builder = StateGraph(AgentState)

In [5]:
from langchain_tavily import TavilySearch

tool = TavilySearch(
    max_results=5,
    topic="general",
    include_answer=True,
    include_raw_content=True,
    include_images=True,
   
)


def web_search(state: AgentState) -> str:
    query = state['query']
    search_results = tool.invoke(query)
    return {'context': search_results}

In [6]:
import langchainhub
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

template = """당신은 질문-답변 과제를 수행하는 인공지능 어시스턴트입니다. 
주어진 문맥(Context)을 사용하여 질문(Question)에 답하세요. 
답을 모른다면 모른다고 답변하세요. 
최대한 세 문장 내로 간결하게 답변하세요.

Question: {question} 
Context: {context} 
Answer:"""

# 직접 프롬프트 객체 생성
generate_prompt = ChatPromptTemplate.from_template(template)
generate_llm = ChatOpenAI(model="gpt-4o")

def web_generate(state:AgentState):
    context = state['context']
    query = state['query']
    generate_chain = generate_prompt | generate_llm | StrOutputParser()
    response = generate_chain.invoke({'question': query, 'context': context})
    return {'answer': response}


In [7]:
basic_llm = ChatOpenAI(model="gpt-4o-mini")

def basic_generate(state: AgentState):
    query = state['query']
    basic_llm_chain = basic_llm | StrOutputParser()
    llm_response = basic_llm_chain.invoke(query)
    return {'answer': llm_response}

In [8]:
#Router 노드로 만들어서 llm 이 판단하여 각 agent 로 분기할수 있도록 

router_prompt = """You are an expert at routing a user's question to 'vector_store', 'llm', or 'web_search'.
'vector_store' contains information about income tax up to December 2024.
if you think the question is simple enough use 'llm'
if you think you need to search the web to answer the question use 'web_search'"""


router_prompt = ChatPromptTemplate.from_messages([
  ('system', router_prompt),
  ('user', '{query}')
])

router_llm = ChatOpenAI(model="gpt-4o-mini")


def router(state: AgentState):
  query = state['query']
  router_chain = router_prompt | router_llm | StrOutputParser()
  route = router_chain.invoke({'query': query})
  print(f'router route == {route}')
  return route




In [9]:
#graph_builder.add_node('router', router)
graph_builder.add_node('web_search', web_search)
graph_builder.add_node('web_generate', web_generate)
graph_builder.add_node('basic_generate', basic_generate)
